<a href="https://colab.research.google.com/github/obaidah3/Automatic-Speech-Recognition/blob/main/asr_training_module2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import auth
from googleapiclient.discovery import build

# Authenticate to Google Colab
auth.authenticate_user()

In [3]:
# Build the Drive service
drive_service = build('drive', 'v3')

# List files from your Google Drive
results = drive_service.files().list(
    pageSize=10, fields="nextPageToken, files(id, name)").execute()
items = results.get('files', [])

if not items:
    print('No files found.')
else:
    print('Files:')
    for item in items:
        print(u'{0} ({1})'.format(item['name'], item['id']))

Files:
asr_training_module2.ipynb (1gMm7SkX_CA0OKqdNuhdzRsPzubnxc3Nu)
ASR_Project (1fnpKwZkHYHdQ05Kk7eVBHevM78TbMTON)
ASR_checkpoints (13A9llGXuiVK7GGCmgU5gw2P_z8vkrSI4)
Colab Notebooks (1ptudDCA64FhRpEKlIzJi294x3J0pcyiN)
best_model.pt (1ovTg1MYxEJmQ3eda87a4J_xJ3yZZtt4r)
last_checkpoint.pt (1ZUgsEBcDoGyIrf1vbUIPXnFcPkPVOffY)
history.json (16QnHgqEfb6wHdGyoSeAu6QmBvq51pCec)
training_state.json (1ysVH292-6KNPd7rPOpgYJa-Hl0vgnA-M)
Waves.7z (1mV5gHb_j9EwQsPMX4CNP_JClsWGYPp_L)
resample_errors.csv (1-nLSmftZf1C5qlLNoSFO22XzdDuinPJS)


In [4]:
import io
from googleapiclient.http import MediaIoBaseDownload

file_id = '1mV5gHb_j9EwQsPMX4CNP_JClsWGYPp_L'
file_name = 'Waves.7z'

request = drive_service.files().get_media(fileId=file_id)
fh = io.FileIO(file_name, 'wb')
downloader = MediaIoBaseDownload(fh, request)
done = False
while done is False:
    status, done = downloader.next_chunk()
    print(f'Download {int(status.progress() * 100)}%.')
print(f'Downloaded {file_name}.')


Download 1%.
Download 3%.
Download 4%.
Download 6%.
Download 8%.
Download 9%.
Download 11%.
Download 13%.
Download 14%.
Download 16%.
Download 17%.
Download 19%.
Download 21%.
Download 22%.
Download 24%.
Download 26%.
Download 27%.
Download 29%.
Download 31%.
Download 32%.
Download 34%.
Download 35%.
Download 37%.
Download 39%.
Download 40%.
Download 42%.
Download 44%.
Download 45%.
Download 47%.
Download 48%.
Download 50%.
Download 52%.
Download 53%.
Download 55%.
Download 57%.
Download 58%.
Download 60%.
Download 62%.
Download 63%.
Download 65%.
Download 66%.
Download 68%.
Download 70%.
Download 71%.
Download 73%.
Download 75%.
Download 76%.
Download 78%.
Download 79%.
Download 81%.
Download 83%.
Download 84%.
Download 86%.
Download 88%.
Download 89%.
Download 91%.
Download 93%.
Download 94%.
Download 96%.
Download 97%.
Download 99%.
Download 100%.
Downloaded Waves.7z.


In [9]:
# Extract the waves.7z file
!7z x Waves.7z -o/content/


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 6425925252 bytes (6129 MiB)

Extracting archive: Waves.7z
--
Path = Waves.7z
Type = 7z
Physical Size = 6425925252
Headers Size = 1148616
Method = LZMA:23
Solid = +
Blocks = 1

  0%      0% - waves/common_voice_ar_19058307.wav                                           0% 35 - waves/common_voice_ar_19058435.wav                                              0% 71 - waves/common_voice_ar_19058507.wav                                              0% 104 - waves/

In [5]:
# Install p7zip-full for extracting .7z files
get_ipython().system('apt-get update && apt-get install -y p7zip-full')


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,965 kB]
Hit:13 https://ppa.launchpadcontent.net

In [42]:
# LOCAL (runtime)
LOCAL_CHECKPOINT_DIR = '/content/checkpoints'
LOCAL_DATA_DIR = '/content/data_cache'
LOCAL_WAV_DIR = '/content/waves'

os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_WAV_DIR, exist_ok=True)

# DRIVE (storage)
WAVES_ID = '1mV5gHb_j9EwQsPMX4CNP_JClsWGYPp_L'

TRAIN_ID = '1uI1egA7yFYpjtxaSLILrYC5R1oTdoubx'
VAL_ID   = '1RaZX4GnbzhvUZedgE3lP3nDxsfGtdZkX'
TEST_ID  = '1PyFg5dxRLbujTjxM2bZ7NUpZzquNJKqr'

LAST_CKPT_ID = '1ZUgsEBcDoGyIrf1vbUIPXnFcPkPVOffY'
BEST_CKPT_ID = '1ovTg1MYxEJmQ3eda87a4J_xJ3yZZtt4r'
HISTORY_ID   = '16QnHgqEfb6wHdGyoSeAu6QmBvq51pCec'
STATE_ID     = '1ysVH292-6KNPd7rPOpgYJa-Hl0vgnA-M'

CHECKPOINTS_FOLDER_ID = '13A9llGXuiVK7GGCmgU5gw2P_z8vkrSI4'


In [45]:
# WAV
#download_file_by_id(drive_service, WAVES_ID, '/content/Waves.7z')

# CSVs
download_file_by_id(drive_service, TRAIN_ID, '/content/train.csv')
download_file_by_id(drive_service, VAL_ID, '/content/val.csv')
download_file_by_id(drive_service, TEST_ID, '/content/test.csv')

# Checkpoints
download_file_by_id(drive_service, LAST_CKPT_ID, f'{LOCAL_CHECKPOINT_DIR}/last_checkpoint.pt')
download_file_by_id(drive_service, BEST_CKPT_ID, f'{LOCAL_CHECKPOINT_DIR}/best_model.pt')
download_file_by_id(drive_service, HISTORY_ID, f'{LOCAL_CHECKPOINT_DIR}/history.json')
download_file_by_id(drive_service, STATE_ID, f'{LOCAL_CHECKPOINT_DIR}/training_state.json')

Downloaded: /content/train.csv
Downloaded: /content/val.csv
Downloaded: /content/test.csv
Downloaded: /content/checkpoints/last_checkpoint.pt
Downloaded: /content/checkpoints/best_model.pt
Downloaded: /content/checkpoints/history.json
Downloaded: /content/checkpoints/training_state.json


In [46]:
import os
import re
import ast
import json
import pandas as pd
from googleapiclient.http import MediaFileUpload

# =========================================================
# CONFIG
# =========================================================
LOCAL_DATA_DIR = '/content/data_cache'
AUDIO_DIR = '/content/waves'
CHECKPOINTS_FOLDER_ID = '13A9llGXuiVK7GGCmgU5gw2P_z8vkrSI4'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

train_csv_path = '/content/train.csv'
val_csv_path = '/content/val.csv'
test_csv_path = '/content/test.csv'

train_clean_path = os.path.join(LOCAL_DATA_DIR, 'train_clean.csv')
val_clean_path = os.path.join(LOCAL_DATA_DIR, 'val_clean.csv')
test_clean_path = os.path.join(LOCAL_DATA_DIR, 'test_clean.csv')

train_ready_path = os.path.join(LOCAL_DATA_DIR, 'train_ready.csv')
val_ready_path = os.path.join(LOCAL_DATA_DIR, 'val_ready.csv')
test_ready_path = os.path.join(LOCAL_DATA_DIR, 'test_ready.csv')

char2idx_path = os.path.join(LOCAL_DATA_DIR, 'char2idx.json')
idx2char_path = os.path.join(LOCAL_DATA_DIR, 'idx2char.json')

# =========================================================
# HELPERS
# =========================================================
def upload_file_to_drive(drive_service, local_path, parent_folder_id, file_name=None):
    if file_name is None:
        file_name = os.path.basename(local_path)

    query = (
        f"name = '{file_name}' and "
        f"'{parent_folder_id}' in parents and "
        "trashed = false"
    )

    results = drive_service.files().list(
        q=query,
        fields="files(id, name)"
    ).execute()

    existing_files = results.get('files', [])
    media = MediaFileUpload(local_path, resumable=True)

    if existing_files:
        file_id = existing_files[0]['id']
        drive_service.files().update(
            fileId=file_id,
            media_body=media
        ).execute()
        print(f"Updated on Drive: {file_name}")
        return file_id
    else:
        metadata = {
            'name': file_name,
            'parents': [parent_folder_id]
        }
        created = drive_service.files().create(
            body=metadata,
            media_body=media,
            fields='id,name'
        ).execute()
        print(f"Uploaded to Drive: {created['name']} ({created['id']})")
        return created['id']


def parse_label_ids(x):
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)


# =========================================================
# 1) LOAD ORIGINAL CSVs
# =========================================================
train_df = pd.read_csv(train_csv_path)
val_df = pd.read_csv(val_csv_path)
test_df = pd.read_csv(test_csv_path)

print("Original shapes:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# =========================================================
# 2) ARABIC TEXT NORMALIZATION
# =========================================================
arabic_diacritics = re.compile(r'[\u064B-\u0652\u0670\u0640]')

def normalize_arabic(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # remove diacritics + tatweel
    text = re.sub(arabic_diacritics, '', text)

    # normalize common letters
    text = re.sub(r'[أإآٱ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)

    # keep Arabic letters, digits, and spaces only
    text = re.sub(r'[^0-9ء-ي ]+', ' ', text)

    # collapse spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

for df in [train_df, val_df, test_df]:
    df['normalized_text'] = df['text'].apply(normalize_arabic)

print("\nSample normalized text:")
print(train_df[['text', 'normalized_text']].head(10))

# =========================================================
# 3) REMOVE EMPTY TEXTS
# =========================================================
train_df = train_df[train_df['normalized_text'].str.len() > 0].reset_index(drop=True)
val_df = val_df[val_df['normalized_text'].str.len() > 0].reset_index(drop=True)
test_df = test_df[test_df['normalized_text'].str.len() > 0].reset_index(drop=True)

print("\nAfter removing empty texts:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# =========================================================
# 4) BUILD VOCAB FROM TRAIN ONLY
# =========================================================
all_text = " ".join(train_df['normalized_text'].astype(str).tolist())
vocab_chars = sorted(list(set(all_text)))

char2idx = {'<blank>': 0}
for i, ch in enumerate(vocab_chars, start=1):
    char2idx[ch] = i

idx2char = {idx: ch for ch, idx in char2idx.items()}

print("\nVocabulary characters:")
print(vocab_chars)
print("Vocab size without blank:", len(vocab_chars))
print("Total vocab size with blank:", len(char2idx))

print("\nSample mapping:")
for k in list(char2idx.keys())[:20]:
    print(repr(k), "->", char2idx[k])

# =========================================================
# 5) ENCODE TEXT TO IDS
# =========================================================
def text_to_ids(text, mapping):
    return [mapping[ch] for ch in text if ch in mapping]

for df in [train_df, val_df, test_df]:
    df['label_ids'] = df['normalized_text'].apply(lambda x: text_to_ids(x, char2idx))

print("\nEncoded sample:")
print(train_df[['normalized_text', 'label_ids']].head(3))

# =========================================================
# 6) SAVE CLEAN CSVs
# =========================================================
train_df.to_csv(train_clean_path, index=False, encoding='utf-8-sig')
val_df.to_csv(val_clean_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_clean_path, index=False, encoding='utf-8-sig')

print("\nSaved clean CSVs:")
print(train_clean_path)
print(val_clean_path)
print(test_clean_path)

# =========================================================
# 7) RELOAD CLEAN CSVs AND FIX AUDIO PATHS
# =========================================================
train_df = pd.read_csv(train_clean_path)
val_df = pd.read_csv(val_clean_path)
test_df = pd.read_csv(test_clean_path)

train_df['resampled_path'] = train_df['file_name'].apply(lambda x: os.path.join(AUDIO_DIR, x))
val_df['resampled_path'] = val_df['file_name'].apply(lambda x: os.path.join(AUDIO_DIR, x))
test_df['resampled_path'] = test_df['file_name'].apply(lambda x: os.path.join(AUDIO_DIR, x))

print("\nFixed audio paths sample:")
print(train_df[['file_name', 'resampled_path']].head())

# =========================================================
# 8) CHECK FILE EXISTENCE
# =========================================================
train_exists = train_df['resampled_path'].apply(os.path.exists).sum()
val_exists = val_df['resampled_path'].apply(os.path.exists).sum()
test_exists = test_df['resampled_path'].apply(os.path.exists).sum()

print(f"\nTrain existing files: {train_exists}/{len(train_df)}")
print(f"Val existing files  : {val_exists}/{len(val_df)}")
print(f"Test existing files : {test_exists}/{len(test_df)}")

train_df = train_df[train_df['resampled_path'].apply(os.path.exists)].reset_index(drop=True)
val_df = val_df[val_df['resampled_path'].apply(os.path.exists)].reset_index(drop=True)
test_df = test_df[test_df['resampled_path'].apply(os.path.exists)].reset_index(drop=True)

print("\nAfter dropping missing files:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# =========================================================
# 9) SAVE READY CSVs
# =========================================================
train_df.to_csv(train_ready_path, index=False, encoding='utf-8-sig')
val_df.to_csv(val_ready_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_ready_path, index=False, encoding='utf-8-sig')

print("\nSaved ready CSVs:")
print(train_ready_path)
print(val_ready_path)
print(test_ready_path)

# =========================================================
# 10) SAVE VOCAB JSONs
# =========================================================
with open(char2idx_path, 'w', encoding='utf-8') as f:
    json.dump(char2idx, f, ensure_ascii=False, indent=2)

idx2char_json = {str(k): v for k, v in idx2char.items()}
with open(idx2char_path, 'w', encoding='utf-8') as f:
    json.dump(idx2char_json, f, ensure_ascii=False, indent=2)

print("\nSaved vocab files:")
print(char2idx_path)
print(idx2char_path)

# =========================================================
# 11) QUICK VALIDATION
# =========================================================
train_check = pd.read_csv(train_ready_path)
train_check['label_ids'] = train_check['label_ids'].apply(parse_label_ids)

print("\nQuick validation:")
print("Train ready shape:", train_check.shape)
print("Vocab size:", len(char2idx))
print(train_check[['normalized_text', 'label_ids']].head(2))

# =========================================================
# 12) UPLOAD PROCESSED FILES TO DRIVE
# =========================================================
upload_file_to_drive(drive_service, train_ready_path, CHECKPOINTS_FOLDER_ID, 'train_ready.csv')
upload_file_to_drive(drive_service, val_ready_path, CHECKPOINTS_FOLDER_ID, 'val_ready.csv')
upload_file_to_drive(drive_service, test_ready_path, CHECKPOINTS_FOLDER_ID, 'test_ready.csv')
upload_file_to_drive(drive_service, char2idx_path, CHECKPOINTS_FOLDER_ID, 'char2idx.json')
upload_file_to_drive(drive_service, idx2char_path, CHECKPOINTS_FOLDER_ID, 'idx2char.json')

print("\nAll preprocessing artifacts uploaded successfully.")

Original shapes:
Train: (62973, 5)
Val  : (7872, 5)
Test : (7872, 5)

Sample normalized text:
                                      text  \
0                             ذلك قلم طويل   
1                    جلبت ليلي انتباه فاضل   
2                        لا تلجا الي العنف   
3              والاجتماع علي المناظرة ابلغ   
4                    واحدهما اغلب من الاخر   
5    انني مدينة لكم بالعرفان عن كل مافعلتم   
6                           القبايل مسلمون   
7                      روي شريك بن ابي نمر   
8  علامة الكذاب جوده باليمين من غير مستحلف   
9                      مجددا؟ ليس مرة اخري   

                           normalized_text  
0                             ذلك قلم طويل  
1                    جلبت ليلي انتباه فاضل  
2                        لا تلجا الي العنف  
3              والاجتماع علي المناظرة ابلغ  
4                    واحدهما اغلب من الاخر  
5    انني مدينة لكم بالعرفان عن كل مافعلتم  
6                           القبايل مسلمون  
7                      روي شريك بن ابي 

In [47]:
import os
from pathlib import Path

def print_tree(root, max_depth=2, max_entries_per_dir=50):
    root = Path(root)
    print(f"\n📁 Tree for: {root}")

    if not root.exists():
        print("❌ Path does not exist")
        return

    def _walk(path, prefix="", depth=0):
        if depth > max_depth:
            return

        try:
            entries = sorted(list(path.iterdir()), key=lambda p: (p.is_file(), p.name.lower()))
        except PermissionError:
            print(prefix + "⛔ Permission denied")
            return

        if len(entries) > max_entries_per_dir:
            shown = entries[:max_entries_per_dir]
            hidden_count = len(entries) - max_entries_per_dir
        else:
            shown = entries
            hidden_count = 0

        for i, entry in enumerate(shown):
            connector = "└── " if i == len(shown) - 1 and hidden_count == 0 else "├── "
            print(prefix + connector + entry.name)

            if entry.is_dir():
                extension = "    " if connector == "└── " else "│   "
                _walk(entry, prefix + extension, depth + 1)

        if hidden_count > 0:
            print(prefix + f"└── ... ({hidden_count} more entries)")

    _walk(root)

# افحص أهم الأماكن
print_tree('/content', max_depth=2, max_entries_per_dir=40)
print_tree('/content/checkpoints', max_depth=2, max_entries_per_dir=40)
print_tree('/content/data_cache', max_depth=2, max_entries_per_dir=40)
print_tree('/content/waves', max_depth=1, max_entries_per_dir=20)


📁 Tree for: /content
├── .config
│   ├── configurations
│   │   └── config_default
│   ├── logs
│   │   ├── 2026.03.30
│   │   └── 2026.04.12
│   ├── .last_opt_in_prompt.yaml
│   ├── .last_survey_prompt.yaml
│   ├── .last_update_check.json
│   ├── active_config
│   ├── config_sentinel
│   ├── default_configs.db
│   └── hidden_gcloud_config_universe_descriptor_data_cache_configs.db
├── checkpoints
│   ├── best_model.pt
│   ├── history.json
│   ├── last_checkpoint.pt
│   └── training_state.json
├── data_cache
│   ├── char2idx.json
│   ├── idx2char.json
│   ├── test_clean.csv
│   ├── test_ready.csv
│   ├── train_clean.csv
│   ├── train_ready.csv
│   ├── val_clean.csv
│   └── val_ready.csv
├── drive
│   └── MyDrive
│       └── ASR_checkpoints
├── sample_data
│   ├── anscombe.json
│   ├── california_housing_test.csv
│   ├── california_housing_train.csv
│   ├── mnist_test.csv
│   ├── mnist_train_small.csv
│   └── README.md
├── waves
│   ├── common_voice_ar_19058307.wav
│   ├── common_voice_

In [49]:
ASR_PROJECT_FOLDER_ID = '15SbBXgQzIZl-P9KbeoMcrtfZ9qr7c3v0'
ASR_CHECKPOINTS_FOLDER_ID = '1hHW83sAHT65AYKSDQsdBR1Ne-9jTKFqz'

def list_drive_folder_tree(drive_service, folder_id, folder_name="ROOT", indent=0, max_depth=2):
    prefix = "    " * indent
    print(f"{prefix}📁 {folder_name} ({folder_id})")

    if indent >= max_depth:
        return

    query = f"'{folder_id}' in parents and trashed = false"
    results = drive_service.files().list(
        q=query,
        pageSize=200,
        fields="files(id, name, mimeType, size)"
    ).execute()

    items = sorted(results.get('files', []), key=lambda x: (x['mimeType'] != 'application/vnd.google-apps.folder', x['name'].lower()))

    for item in items:
        name = item['name']
        file_id = item['id']
        mime = item['mimeType']
        size = item.get('size', 'N/A')

        if mime == 'application/vnd.google-apps.folder':
            list_drive_folder_tree(drive_service, file_id, folder_name=name, indent=indent+1, max_depth=max_depth)
        else:
            print(f"{prefix}    📄 {name} ({file_id}) | size={size} | mime={mime}")

# افحص فولدرات الدرايف
list_drive_folder_tree(drive_service, ASR_PROJECT_FOLDER_ID, folder_name="ASR_Project", max_depth=2)
print()
list_drive_folder_tree(drive_service, ASR_CHECKPOINTS_FOLDER_ID, folder_name="ASR_checkpoints", max_depth=2)

📁 ASR_Project (15SbBXgQzIZl-P9KbeoMcrtfZ9qr7c3v0)
    📄 resample_errors.csv (1v75kCKN7vtNk32_dN3jnxGhqhdVEhjwz) | size=1 | mime=text/csv
    📄 resample_success.csv (1lqrCKtM_N0WvyQIjz43EPmAfyn0J7HNZ) | size=11972020 | mime=text/csv
    📄 test.csv (1YMVTT9yyLmRFwMzJQo-xrCIAxjZWKKfo) | size=1610989 | mime=text/csv
    📄 train.csv (18ohUcPAmoPjc7vkPXywznrcQqd6Rx2Lj) | size=12909773 | mime=text/csv
    📄 val.csv (1C2oSpw3lqxQWJ9tM3vzz1LPZ5l9ifF6z) | size=1614492 | mime=text/csv
    📄 Waves.7z (17a-fTlFVIpugAFilfuCFUXrHgaIUD1w5) | size=6425925252 | mime=application/x-7z-compressed

📁 ASR_checkpoints (1hHW83sAHT65AYKSDQsdBR1Ne-9jTKFqz)
    📄 best_model.pt (18KmG9OlQrK8WK-6nIQBfTugtYUkFpFVN) | size=41355577 | mime=application/x-zip
    📄 history.json (1uVhBHuscQYG2a1Xai4h2slnnrtddRQqQ) | size=424 | mime=application/json
    📄 last_checkpoint.pt (1Nebz8KYeRtyfY5mim2qOPGNRNUU5hmW6) | size=41356447 | mime=application/x-zip
    📄 training_state.json (1hwm5EHRCMDYzZAzfE2y3vGStbO6uQm1Q) | size=487 

In [51]:
import os

important_local_files = [
    '/content/train.csv',
    '/content/val.csv',
    '/content/test.csv',
    '/content/Waves.7z',
    '/content/checkpoints/last_checkpoint.pt',
    '/content/checkpoints/best_model.pt',
    '/content/checkpoints/history.json',
    '/content/checkpoints/training_state.json',
    '/content/data_cache/train_ready.csv',
    '/content/data_cache/val_ready.csv',
    '/content/data_cache/test_ready.csv',
    '/content/data_cache/char2idx.json',
    '/content/data_cache/idx2char.json',
]

print("\n📌 Important local files status:")
for path in important_local_files:
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / (1024**2) if exists and os.path.isfile(path) else 0
    print(f"{'✅' if exists else '❌'} {path}" + (f" | {size_mb:.2f} MB" if exists and os.path.isfile(path) else ""))

    important_drive_names = [
    'train.csv', 'val.csv', 'test.csv',
    'Waves.7z',
    'last_checkpoint.pt', 'best_model.pt',
    'history.json', 'training_state.json',
    'train_ready.csv', 'val_ready.csv', 'test_ready.csv',
    'char2idx.json', 'idx2char.json'
]

print("\n📌 Important Drive files status:")
for fname in important_drive_names:
    query = f"name = '{fname}' and trashed = false"
    results = drive_service.files().list(
        q=query,
        pageSize=20,
        fields="files(id, name, mimeType, size)"
    ).execute()
    items = results.get('files', [])

    if items:
        print(f"✅ {fname}")
        for item in items:
            print(f"   ↳ ID={item['id']} | size={item.get('size', 'N/A')} | mime={item['mimeType']}")
    else:
        print(f"❌ {fname}")


📌 Important local files status:
✅ /content/train.csv | 12.31 MB
✅ /content/val.csv | 1.54 MB
✅ /content/test.csv | 1.54 MB
✅ /content/Waves.7z | 100.00 MB
✅ /content/checkpoints/last_checkpoint.pt | 39.44 MB
✅ /content/checkpoints/best_model.pt | 39.44 MB
✅ /content/checkpoints/history.json | 0.00 MB
✅ /content/checkpoints/training_state.json | 0.00 MB
✅ /content/data_cache/train_ready.csv | 20.67 MB
✅ /content/data_cache/val_ready.csv | 2.59 MB
✅ /content/data_cache/test_ready.csv | 2.57 MB
✅ /content/data_cache/char2idx.json | 0.00 MB
✅ /content/data_cache/idx2char.json | 0.00 MB

📌 Important Drive files status:
✅ train.csv
   ↳ ID=18ohUcPAmoPjc7vkPXywznrcQqd6Rx2Lj | size=12909773 | mime=text/csv
   ↳ ID=1uI1egA7yFYpjtxaSLILrYC5R1oTdoubx | size=12909773 | mime=text/csv
✅ val.csv
   ↳ ID=1C2oSpw3lqxQWJ9tM3vzz1LPZ5l9ifF6z | size=1614492 | mime=text/csv
   ↳ ID=1RaZX4GnbzhvUZedgE3lP3nDxsfGtdZkX | size=1614492 | mime=text/csv
✅ test.csv
   ↳ ID=1YMVTT9yyLmRFwMzJQo-xrCIAxjZWKKfo | size=1

In [52]:
import pandas as pd

def inspect_csv(path, name):
    print(f"\n📊 Inspecting: {name}")
    if not os.path.exists(path):
        print("❌ File not found")
        return

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nDtypes:")
    print(df.dtypes)
    print("\nNull counts:")
    print(df.isnull().sum())
    print("\nSample rows:")
    print(df.head(3))

inspect_csv('/content/train.csv', 'train.csv')
inspect_csv('/content/val.csv', 'val.csv')
inspect_csv('/content/test.csv', 'test.csv')

inspect_csv('/content/data_cache/train_ready.csv', 'train_ready.csv')
inspect_csv('/content/data_cache/val_ready.csv', 'val_ready.csv')
inspect_csv('/content/data_cache/test_ready.csv', 'test_ready.csv')


📊 Inspecting: train.csv
Shape: (62973, 5)
Columns: ['text', 'file_name', 'audio_path', 'duration', 'resampled_path']

Dtypes:
text               object
file_name          object
audio_path         object
duration          float64
resampled_path     object
dtype: object

Null counts:
text              0
file_name         0
audio_path        0
duration          0
resampled_path    0
dtype: int64

Sample rows:
                    text                     file_name  \
0           ذلك قلم طويل  common_voice_ar_24068997.wav   
1  جلبت ليلي انتباه فاضل  common_voice_ar_24305449.wav   
2      لا تلجا الي العنف  common_voice_ar_19376237.wav   

                                          audio_path  duration  \
0  /kaggle/working/audio_16k/common_voice_ar_2406...  2.664036   
1  /kaggle/working/audio_16k/common_voice_ar_2430...  3.240000   
2  /kaggle/working/audio_16k/common_voice_ar_1937...  2.664036   

                                      resampled_path  
0  /kaggle/working/audio_16k/common

In [53]:
import torch

def inspect_checkpoint(path):
    print(f"\n🧩 Inspecting checkpoint: {path}")
    if not os.path.exists(path):
        print("❌ Checkpoint not found")
        return

    ckpt = torch.load(path, map_location='cpu')
    print("Keys:", ckpt.keys())

    if 'epoch' in ckpt:
        print("Last saved epoch:", ckpt['epoch'])
    if 'best_val_loss' in ckpt:
        print("Best val loss:", ckpt['best_val_loss'])
    if 'epochs_without_improvement' in ckpt:
        print("Epochs without improvement:", ckpt['epochs_without_improvement'])
    if 'history' in ckpt:
        hist = ckpt['history']
        print("History keys:", hist.keys())
        print("Train loss count:", len(hist.get('train_loss', [])))
        print("Val loss count  :", len(hist.get('val_loss', [])))

inspect_checkpoint('/content/checkpoints/last_checkpoint.pt')
inspect_checkpoint('/content/checkpoints/best_model.pt')


🧩 Inspecting checkpoint: /content/checkpoints/last_checkpoint.pt
Keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'best_val_loss', 'epochs_without_improvement', 'history'])
Last saved epoch: 7
Best val loss: 0.8038222618340477
Epochs without improvement: 0
History keys: dict_keys(['train_loss', 'val_loss'])
Train loss count: 8
Val loss count  : 8

🧩 Inspecting checkpoint: /content/checkpoints/best_model.pt
Keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'best_val_loss', 'epochs_without_improvement', 'history'])
Last saved epoch: 7
Best val loss: 0.8009154502817286
Epochs without improvement: 0
History keys: dict_keys(['train_loss', 'val_loss'])
Train loss count: 8
Val loss count  : 8


In [54]:
required_before_training = {
    "train.csv": '/content/train.csv',
    "val.csv": '/content/val.csv',
    "test.csv": '/content/test.csv',
    "train_ready.csv": '/content/data_cache/train_ready.csv',
    "val_ready.csv": '/content/data_cache/val_ready.csv',
    "test_ready.csv": '/content/data_cache/test_ready.csv',
    "char2idx.json": '/content/data_cache/char2idx.json',
    "idx2char.json": '/content/data_cache/idx2char.json',
    "waves dir": '/content/waves',
    "last_checkpoint.pt": '/content/checkpoints/last_checkpoint.pt',
}

print("\n✅ Pre-training readiness check:")
all_ok = True
for name, path in required_before_training.items():
    ok = os.path.exists(path)
    all_ok = all_ok and ok
    print(f"{'✅' if ok else '❌'} {name}: {path}")

print("\nFinal status:", "READY FOR TRAINING 🚀" if all_ok else "NOT READY ❌")


✅ Pre-training readiness check:
✅ train.csv: /content/train.csv
✅ val.csv: /content/val.csv
✅ test.csv: /content/test.csv
✅ train_ready.csv: /content/data_cache/train_ready.csv
✅ val_ready.csv: /content/data_cache/val_ready.csv
✅ test_ready.csv: /content/data_cache/test_ready.csv
✅ char2idx.json: /content/data_cache/char2idx.json
✅ idx2char.json: /content/data_cache/idx2char.json
✅ waves dir: /content/waves
✅ last_checkpoint.pt: /content/checkpoints/last_checkpoint.pt

Final status: READY FOR TRAINING 🚀




---



In [55]:
batch = next(iter(train_loader))

print("features shape       :", batch['features'].shape)         # [B, T, n_mels]
print("feature_lengths shape:", batch['feature_lengths'].shape)
print("labels shape         :", batch['labels'].shape)
print("label_lengths shape  :", batch['label_lengths'].shape)

print("\nExample text:")
print(batch['texts'][0])

print("\nExample audio path:")
print(batch['audio_paths'][0])

features shape       : torch.Size([16, 533, 80])
feature_lengths shape: torch.Size([16])
labels shape         : torch.Size([394])
label_lengths shape  : torch.Size([16])

Example text:
وهو الظل الذي يلجا اليه المضطرون والحمي الذي ياوي اليه الخايفون

Example audio path:
/content/waves/common_voice_ar_24160332.wav


In [56]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# مهم: لازم num_classes يساوي حجم الـ vocabulary الحقيقي
num_classes = len(char2idx)

model = CNNBiLSTMCTC(
    num_classes=num_classes,
    n_mels=80,
    proj_dim=256,
    lstm_hidden=256,
    lstm_layers=2,
    dropout=0.2
).to(device)

print(model)

Device: cuda
CNNBiLSTMCTC(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.1, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
    (14): MaxPool2d(kernel_size=(2, 2), stride=(

In [ ]:
import os
import json
import time
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from googleapiclient.http import MediaFileUpload

# =========================================================
# 0) DRIVE CONFIG
# =========================================================
CHECKPOINTS_FOLDER_ID = '13A9llGXuiVK7GGCmgU5gw2P_z8vkrSI4'

def upload_file_to_drive(drive_service, local_path, parent_folder_id, file_name=None):
    if file_name is None:
        file_name = os.path.basename(local_path)

    query = (
        f"name = '{file_name}' and "
        f"'{parent_folder_id}' in parents and "
        "trashed = false"
    )

    results = drive_service.files().list(
        q=query,
        fields="files(id, name)"
    ).execute()

    existing_files = results.get('files', [])
    media = MediaFileUpload(local_path, resumable=True)

    if existing_files:
        file_id = existing_files[0]['id']
        drive_service.files().update(
            fileId=file_id,
            media_body=media
        ).execute()
        print(f"Updated on Drive: {file_name}")
        return file_id
    else:
        metadata = {
            'name': file_name,
            'parents': [parent_folder_id]
        }
        created = drive_service.files().create(
            body=metadata,
            media_body=media,
            fields='id,name'
        ).execute()
        print(f"Uploaded to Drive: {created['name']} ({created['id']})")
        return created['id']

# =========================================================
# 1) CONFIG
# =========================================================
checkpoint_dir = '/content/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

last_ckpt_path = os.path.join(checkpoint_dir, 'last_checkpoint.pt')
best_ckpt_path = os.path.join(checkpoint_dir, 'best_model.pt')
history_path = os.path.join(checkpoint_dir, 'history.json')
training_state_path = os.path.join(checkpoint_dir, 'training_state.json')

epochs_per_run = 3

# Early Stopping
patience = 3
min_delta = 0.001

# Optimization
learning_rate = 1e-3
weight_decay = 1e-5
max_grad_norm = 5.0

# =========================================================
# 2) DEVICE / LOSS / OPTIMIZER / SCALER
# =========================================================
use_cuda = torch.cuda.is_available()
device_type = 'cuda' if use_cuda else 'cpu'

print("Device:", device)
print("Using checkpoint dir:", checkpoint_dir)
print("Last checkpoint exists:", os.path.exists(last_ckpt_path))

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

scaler = torch.amp.GradScaler(device_type, enabled=use_cuda)

# =========================================================
# 3) STATE VARIABLES
# =========================================================
best_val_loss = float('inf')
start_epoch = 0
epochs_without_improvement = 0

history = {
    'train_loss': [],
    'val_loss': []
}

# =========================================================
# 4) HELPERS
# =========================================================
def save_json(path, data):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def save_checkpoint(path, epoch, best_val_loss, epochs_without_improvement, history):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_loss': best_val_loss,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history
    }, path)

def load_checkpoint_if_exists():
    global best_val_loss, start_epoch, epochs_without_improvement, history

    if os.path.exists(last_ckpt_path):
        print(f"Found checkpoint: {last_ckpt_path}")
        ckpt = torch.load(last_ckpt_path, map_location=device)

        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])

        if ckpt.get('scaler_state_dict') is not None:
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        start_epoch = ckpt.get('epoch', -1) + 1
        best_val_loss = ckpt.get('best_val_loss', float('inf'))
        epochs_without_improvement = ckpt.get('epochs_without_improvement', 0)
        history = ckpt.get('history', {'train_loss': [], 'val_loss': []})

        print(f"Resumed from epoch {start_epoch}")
        print(f"Best val loss so far: {best_val_loss:.4f}")
        print(f"epochs_without_improvement: {epochs_without_improvement}")
    else:
        print("No previous checkpoint found. Starting from scratch.")

# =========================================================
# 5) TRAIN / VALIDATE
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    processed_batches = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for batch_idx, batch in enumerate(pbar):
        features = batch['features'].to(device, non_blocking=True)
        feature_lengths = batch['feature_lengths'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        label_lengths = batch['label_lengths'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        try:
            with torch.amp.autocast(device_type=device_type, enabled=use_cuda):
                log_probs, output_lengths = model(features, feature_lengths)

                # CTC expects: (T, B, C)
                log_probs = log_probs.permute(1, 0, 2)

                loss = criterion(
                    log_probs,
                    labels,
                    output_lengths,
                    label_lengths
                )

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            processed_batches += 1
            avg_loss = running_loss / processed_batches

            pbar.set_postfix({
                'train_loss': f'{avg_loss:.4f}',
                'batch_loss': f'{loss.item():.4f}'
            })

        except RuntimeError as e:
            print(f"\nSkipped train batch {batch_idx} بسبب RuntimeError: {e}")
            if use_cuda:
                torch.cuda.empty_cache()
            continue

    if processed_batches == 0:
        return float('inf')

    return running_loss / processed_batches


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    processed_batches = 0

    pbar = tqdm(loader, desc="Validation", leave=False)

    for batch_idx, batch in enumerate(pbar):
        features = batch['features'].to(device, non_blocking=True)
        feature_lengths = batch['feature_lengths'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        label_lengths = batch['label_lengths'].to(device, non_blocking=True)

        try:
            with torch.amp.autocast(device_type=device_type, enabled=use_cuda):
                log_probs, output_lengths = model(features, feature_lengths)
                log_probs = log_probs.permute(1, 0, 2)

                loss = criterion(
                    log_probs,
                    labels,
                    output_lengths,
                    label_lengths
                )

            running_loss += loss.item()
            processed_batches += 1
            avg_loss = running_loss / processed_batches

            pbar.set_postfix({
                'val_loss': f'{avg_loss:.4f}',
                'batch_loss': f'{loss.item():.4f}'
            })

        except RuntimeError as e:
            print(f"\nSkipped val batch {batch_idx} بسبب RuntimeError: {e}")
            if use_cuda:
                torch.cuda.empty_cache()
            continue

    if processed_batches == 0:
        return float('inf')

    return running_loss / processed_batches

# =========================================================
# 6) RESUME
# =========================================================
load_checkpoint_if_exists()

end_epoch = start_epoch + epochs_per_run
print(f"Will train from epoch {start_epoch} to {end_epoch - 1}")

# =========================================================
# 7) TRAINING LOOP WITH EARLY STOPPING + DRIVE UPLOAD
# =========================================================
for epoch in range(start_epoch, end_epoch):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}")
    print(f"{'='*60}")

    epoch_start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss = validate_one_epoch(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    epoch_time_min = (time.time() - epoch_start_time) / 60.0

    improvement = best_val_loss - val_loss if best_val_loss != float('inf') else 0.0
    generalization_gap = val_loss - train_loss

    print(f"Epoch {epoch + 1}:")
    print(f"  train_loss = {train_loss:.4f}")
    print(f"  val_loss   = {val_loss:.4f}")
    print(f"  improvement = {improvement:.6f}")
    print(f"  generalization_gap = {generalization_gap:.4f}")
    print(f"  time = {epoch_time_min:.2f} min")

    # -----------------------------------------
    # Save LAST checkpoint every epoch (local)
    # -----------------------------------------
    save_checkpoint(
        path=last_ckpt_path,
        epoch=epoch,
        best_val_loss=best_val_loss,
        epochs_without_improvement=epochs_without_improvement,
        history=history
    )

    # -----------------------------------------
    # Save JSON files every epoch (local)
    # -----------------------------------------
    save_json(history_path, history)

    save_json(training_state_path, {
        'last_completed_epoch': epoch,
        'next_start_epoch': epoch + 1,
        'best_val_loss': best_val_loss,
        'epochs_without_improvement': epochs_without_improvement,
        'patience': patience,
        'min_delta': min_delta,
        'epochs_per_run': epochs_per_run,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'max_grad_norm': max_grad_norm,
        'last_train_loss': train_loss,
        'last_val_loss': val_loss,
        'last_improvement': improvement,
        'last_generalization_gap': generalization_gap,
        'last_epoch_time_min': epoch_time_min
    })

    # -----------------------------------------
    # Upload core files to Drive after every epoch
    # -----------------------------------------
    upload_file_to_drive(drive_service, last_ckpt_path, CHECKPOINTS_FOLDER_ID, 'last_checkpoint.pt')
    upload_file_to_drive(drive_service, history_path, CHECKPOINTS_FOLDER_ID, 'history.json')
    upload_file_to_drive(drive_service, training_state_path, CHECKPOINTS_FOLDER_ID, 'training_state.json')

    # -----------------------------------------
    # Check improvement
    # -----------------------------------------
    if val_loss < (best_val_loss - min_delta):
        best_val_loss = val_loss
        epochs_without_improvement = 0

        # Save BEST checkpoint (local)
        save_checkpoint(
            path=best_ckpt_path,
            epoch=epoch,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            history=history
        )

        # Re-save LAST checkpoint after updating best_val_loss
        save_checkpoint(
            path=last_ckpt_path,
            epoch=epoch,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            history=history
        )

        # Update state again after new best
        save_json(training_state_path, {
            'last_completed_epoch': epoch,
            'next_start_epoch': epoch + 1,
            'best_val_loss': best_val_loss,
            'epochs_without_improvement': epochs_without_improvement,
            'patience': patience,
            'min_delta': min_delta,
            'epochs_per_run': epochs_per_run,
            'learning_rate': learning_rate,
            'weight_decay': weight_decay,
            'max_grad_norm': max_grad_norm,
            'last_train_loss': train_loss,
            'last_val_loss': val_loss,
            'last_improvement': improvement,
            'last_generalization_gap': generalization_gap,
            'last_epoch_time_min': epoch_time_min
        })

        print(f"Saved BEST model at epoch {epoch + 1} with val_loss = {val_loss:.4f}")

        # Upload updated files to Drive
        upload_file_to_drive(drive_service, last_ckpt_path, CHECKPOINTS_FOLDER_ID, 'last_checkpoint.pt')
        upload_file_to_drive(drive_service, best_ckpt_path, CHECKPOINTS_FOLDER_ID, 'best_model.pt')
        upload_file_to_drive(drive_service, training_state_path, CHECKPOINTS_FOLDER_ID, 'training_state.json')

    else:
        epochs_without_improvement += 1
        print(f"No significant improvement. Counter: {epochs_without_improvement}/{patience}")

        # Save updated counter into LAST checkpoint
        save_checkpoint(
            path=last_ckpt_path,
            epoch=epoch,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            history=history
        )

        save_json(training_state_path, {
            'last_completed_epoch': epoch,
            'next_start_epoch': epoch + 1,
            'best_val_loss': best_val_loss,
            'epochs_without_improvement': epochs_without_improvement,
            'patience': patience,
            'min_delta': min_delta,
            'epochs_per_run': epochs_per_run,
            'learning_rate': learning_rate,
            'weight_decay': weight_decay,
            'max_grad_norm': max_grad_norm,
            'last_train_loss': train_loss,
            'last_val_loss': val_loss,
            'last_improvement': improvement,
            'last_generalization_gap': generalization_gap,
            'last_epoch_time_min': epoch_time_min
        })

        # Upload updated files to Drive
        upload_file_to_drive(drive_service, last_ckpt_path, CHECKPOINTS_FOLDER_ID, 'last_checkpoint.pt')
        upload_file_to_drive(drive_service, training_state_path, CHECKPOINTS_FOLDER_ID, 'training_state.json')

    # -----------------------------------------
    # Early stopping
    # -----------------------------------------
    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}")
        print(f"Best val_loss = {best_val_loss:.4f}")
        break

print("\nTraining session complete.")
print("Saved files:")
print(" -", last_ckpt_path)
print(" -", best_ckpt_path)
print(" -", history_path)
print(" -", training_state_path)

Device: cuda
Using checkpoint dir: /content/checkpoints
Last checkpoint exists: True
Found checkpoint: /content/checkpoints/last_checkpoint.pt
Resumed from epoch 8
Best val loss so far: 0.8038
epochs_without_improvement: 0
Will train from epoch 8 to 10

Epoch 9


Training:   0%|          | 0/3936 [00:00<?, ?it/s]

Validation:   0%|          | 0/492 [00:00<?, ?it/s]

Epoch 9:
  train_loss = 0.6901
  val_loss   = 0.7784
  improvement = 0.025445
  generalization_gap = 0.0883
  time = 23.23 min


Updated on Drive: last_checkpoint.pt
Updated on Drive: history.json
Updated on Drive: training_state.json
Saved BEST model at epoch 9 with val_loss = 0.7784
Updated on Drive: last_checkpoint.pt
Updated on Drive: best_model.pt
Updated on Drive: training_state.json

Epoch 10


Training:   0%|          | 0/3936 [00:00<?, ?it/s]